In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("test")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/ramadhan/dev/sandbox/dez/batch-processing/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/19 13:33:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.parquet("yellow/2009/01/")

In [3]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = true)



In [4]:
df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID")

DataFrame[tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [5]:
df.filter(df.PULocationID == 163)

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, passenger_count: int, trip_distance: float, RatecodeID: int, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: int, fare_amount: float, extra: float, mta_tax: float, tip_amount: float, tolls_amount: float, improvement_surcharge: float, total_amount: float, congestion_surcharge: float]

In [6]:
df.take(5)

[Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2019, 1, 3, 10, 22, 27), tpep_dropoff_datetime=datetime.datetime(2019, 1, 3, 10, 24, 39), passenger_count=1, trip_distance=0.30000001192092896, RatecodeID=1, store_and_fwd_flag='N', PULocationID=237, DOLocationID=141, payment_type=1, fare_amount=3.5, extra=0.0, mta_tax=0.5, tip_amount=1.25, tolls_amount=0.0, improvement_surcharge=0.30000001192092896, total_amount=5.550000190734863, congestion_surcharge=None),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2019, 1, 2, 1, 29, 39), tpep_dropoff_datetime=datetime.datetime(2019, 1, 2, 1, 33, 31), passenger_count=1, trip_distance=0.8299999833106995, RatecodeID=1, store_and_fwd_flag='N', PULocationID=263, DOLocationID=75, payment_type=1, fare_amount=5.0, extra=0.5, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=0.30000001192092896, total_amount=6.300000190734863, congestion_surcharge=None),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2019, 1, 1,

In [7]:
from pyspark.sql import functions as F

In [8]:
(
    df
    .withColumn("pickup_date", F.to_date(df.tpep_pickup_datetime))
    .withColumn("dropoff_date", F.to_date(df.tpep_dropoff_datetime))
    .show()
)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|pickup_date|dropoff_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+
|       1| 2019-01-03 10:22:27|  2019-01-03 10:24:39|              1|          0.3|         1|                 N|         237|         141|         

In [9]:
(
    df
    .withColumn("pickup_date", F.to_date(df.tpep_pickup_datetime))
    .withColumn("dropoff_date", F.to_date(df.tpep_dropoff_datetime))
    .select("pickup_date", "dropoff_date", "PULocationID", "DOLocationID")
    .show()
)

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2019-01-03|  2019-01-03|         237|         141|
| 2019-01-02|  2019-01-02|         263|          75|
| 2019-01-01|  2019-01-01|         163|         237|
| 2019-01-02|  2019-01-02|         132|         223|
| 2019-01-03|  2019-01-03|         161|         113|
| 2019-01-03|  2019-01-03|         238|         143|
| 2019-01-01|  2019-01-01|         158|         228|
| 2019-01-01|  2019-01-01|         163|         100|
| 2019-01-02|  2019-01-02|          68|          48|
| 2019-01-03|  2019-01-03|         137|         229|
| 2019-01-02|  2019-01-02|         132|         138|
| 2019-01-02|  2019-01-02|         230|         229|
| 2019-01-03|  2019-01-03|         107|         263|
| 2019-01-04|  2019-01-04|         148|         145|
| 2019-01-02|  2019-01-02|         237|         237|
| 2019-01-04|  2019-01-04|         162|       

In [16]:
def crazy_stuff(num):
    num = str(num)
    num = int(num[1:])
    if num % 7 == 0:
        return f"s/{num:03x}"
    elif num % 3 == 0:
        return f"a/{num:03x}"
    else: 
        return f"e/{num:03x}"


crazy_stuff("B02884")

's/b44'

In [17]:
from pyspark.sql import types

crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

/Users/ramadhan/dev/sandbox/dez/batch-processing/.venv/lib/python3.14/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [18]:
(
    df
    .withColumn("pickup_date", F.to_date(df.tpep_pickup_datetime))
    .withColumn("dropoff_date", F.to_date(df.tpep_dropoff_datetime))
    .withColumn("base_id", crazy_stuff_udf(df.PULocationID))
    .select("base_id", "pickup_date", "dropoff_date", "PULocationID", "DOLocationID")
    .show()
)

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/025| 2019-01-03|  2019-01-03|         237|         141|
|  s/03f| 2019-01-02|  2019-01-02|         263|          75|
|  s/03f| 2019-01-01|  2019-01-01|         163|         237|
|  e/020| 2019-01-02|  2019-01-02|         132|         223|
|  e/03d| 2019-01-03|  2019-01-03|         161|         113|
|  e/026| 2019-01-03|  2019-01-03|         238|         143|
|  e/03a| 2019-01-01|  2019-01-01|         158|         228|
|  s/03f| 2019-01-01|  2019-01-01|         163|         100|
|  e/008| 2019-01-02|  2019-01-02|          68|          48|
|  e/025| 2019-01-03|  2019-01-03|         137|         229|
|  e/020| 2019-01-02|  2019-01-02|         132|         138|
|  a/01e| 2019-01-02|  2019-01-02|         230|         229|
|  s/007| 2019-01-03|  2019-01-03|         107|         263|
|  a/030| 2019-01-04|  2